In [2]:
import datetime as dt
from datetime import datetime, timedelta
import dask
import numpy as np
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

import xarray as xr
from glob import glob
from time import time
import warnings

import pandas as pd

warnings.simplefilter("ignore")

# Set Parameters

# Plots
- Maps
    - [ ] Monats Aufenthaltswahrscheinlichkeiten
    - [ ] Jahres Aufenthaltswahrscheinlichkeiten
    - [ ] Linen zwischen Stationen
- Connectivitätsmatrix
    - [ ] Monats Averages (logscale)
    - [ ] Jahres Averages (logscale)
    - [ ] Einzelne Jahre

In [3]:
# Parameters
site_counter = np.arange(16)
years = np.arange(2016,2025)
year = 2023
nbins = 100
isPapermill = True
start_site_counter = 10
samples = [
    "Speicherkoog", "KB03", "SW08",
    "Wilhelmshaven", "GB96", "H21",
    "Boknis_Eck", "Gdynia", "Riga",
    "IU7_c", "LL17_c", "Finland",
    "BB23", "BB36", "Speicherkoog_Außen",
    "Wilhelmshaven_Außen",
]
lats = [
    54.0929, 54.692, 54.413,
    53.513, 56.54, 54.666,
    54.5169, 54.5829, 57.3911,
    59.489, 59.02, 59.77,
    55.1751, 54.575, 54.11,
    53.637,
]
lons = [
    8.9487, 10.1035, 11.617,
    8.149, 19.5809, 13.0224,
    10.034, 18.6042, 23.8588,
    21.2021, 21.0478, 23.2663,
    15.4352, 16.001, 8.5,
    8.15,
]

number_sites = len(samples)
number_years = len(years)
dist = 20

# Functions

In [4]:
def connectivity_function(
    ds=None,
    dist=None,
):
    # Mean radius of the Earth in km
    R = 6371  

    # Stations (targets)
    obs_lon_deg = ds.lon
    obs_lat_deg = ds.lat
    obs_lon_km = R * np.cos(np.radians(obs_lat_deg)) * np.radians(obs_lon_deg)
    obs_lat_km = R * np.radians(obs_lat_deg)

    # Particles
    target_lon_deg = xr.DataArray(lons)
    target_lat_deg = xr.DataArray(lats)
    target_lon_km = R * np.cos(np.radians(target_lat_deg)) * np.radians(target_lon_deg)
    target_lat_km = R * np.radians(target_lat_deg)

    # Calculate distance to target
    distance_to_target = (
        ((obs_lon_km - target_lon_km) ** 2) + ((obs_lat_km - target_lat_km) ** 2)
    ) ** 0.5

    # Return number of particles within a certain distance from target
    return (distance_to_target <= dist).sum(dim=["obs", "trajectory"]).compute().values
    # return distance_to_target

In [ ]:
trajectory_path = "/gxfs_work/geomar/smomw597/2025_copepods/output/Trajectories/"
out_path_matrix = "/gxfs_work/geomar/smomw597/2025_copepods/output/Connectivity/"
out_path_hist = "/gxfs_work/geomar/smomw597/2025_copepods/output/hist/"

In [6]:
ds_trajectories = xr.Dataset()
monthly_connectivity_matrices = {}
export_vars = ["lat", "lon", "time", "S", "T", "eta", "z", "h0", "age_sec"]
export_vars = ["lat", "lon", "time", "age_sec"]
if isPapermill == False:
    month_split = [None, 4,9,13,17,22]
else:
    month_split = [None, 30000,61000,92000,122000,153000]
expand_dims_dict = {"site":[start_site_counter], "year":[year],}

In [7]:
lonbins = np.arange(8, 12.25, 0.25)
latbins = np.arange(52, 57.25, 0.25)
print(latbins.shape, lonbins.shape)

(21,) (17,)


In [8]:
# age_data_arrays = []
# age_arrays = []
# # for site in np.array([1, 2, 6,]):
# for site in np.arange(16):
#     file = Path(
#         trajectory_path +
#         f"PPmill_Nested_{year}0601-{year}1101_dt15min_site{site:02d}_d0m-25m_N1000_seed123.zarr"
#     )
#     ds_trajectories = xr.open_zarr(file)
#     ds_bins = (ds_trajectories
#         .isel(trajectory=slice(days[day_id], day))
#         .get(["lat", "lon", "age_sec"])
#         .groupby_bins(
#             "age_sec", 
#             np.arange(0,29*(60*60*24),(60*60*24)),
#             labels=np.arange(1,29)
#         )
#     )
#     age_arrays = []
#     for age in np.arange(1,29):
#         test_lons_array = ds_bins[age].lon.values
#         test_lats_array = ds_bins[age].lat.values
#         hist, xedges, yedges = np.histogram2d(
#             test_lons_array, test_lats_array,
#             bins=(lonbins, latbins),
#         )
#         da = xr.DataArray(
#             hist,
#             dims={
#                 "lon":lonbins_da,
#                 "lat":latbins_da,
#             },
#             coords={
#                 "lon":lonbins_da,
#                 "lat":latbins_da,
#             },
#             name=f"{age}-{site}",
#         ).expand_dims({
#             "site":[site],
#             "age": [age],
#             }
#         )
#         age_arrays.append(da)
#     age_data = xr.concat(age_arrays, dim="age")
#     age_data_arrays.append(age_data)
# ds = xr.concat(age_data_arrays, dim="site")

In [9]:
lonbins = np.arange(0, 30.25, 0.25)
latbins = np.arange(52, 62.25, 0.25)
print(latbins.shape, lonbins.shape)

(41,) (121,)


In [ ]:
trajectories = np.arange(0, 153001, 1000)
# trajectories = np.arange(150000, 153001, 1000)
# trajectories = np.arange(100000, 105001, 1000)
# for day_id, day in enumerate(days[1:]):
#     print(day-1000,day)

In [17]:
lonbins_da = (lonbins[1:]+lonbins[:-1])/2
latbins_da = (latbins[1:]+latbins[:-1])/2

age_data_arrays = []
age_arrays = []
data_arrays = []
# for site in np.array([1, 2, 6,]):
for site in np.arange(16):
    file = Path(
        trajectory_path +
        f"PPmill_Nested_{year}0601-{year}1101_dt15min_site{site:02d}_d0m-25m_N1000_seed123.zarr"
    )
    ds_trajectories = xr.open_zarr(file)
    daily_arrays = []
    for traj in trajectories[:-1]:
        start_day = int(np.floor(traj/1000))
        ds_bins = (ds_trajectories
            .isel(trajectory=slice(traj,traj+1000))
            .get(["lat", "lon", "age_sec", "time"])
            .groupby_bins(
                "age_sec", 
                np.arange(0,29*(60*60*24),(60*60*24)),
                labels=np.arange(1,29)
            )
        )
        age_arrays = []
        for age in np.arange(1,29):
        # for age in np.arange(1,5):
            try:
                test_lons_array = ds_bins[age].lon.values
                test_lats_array = ds_bins[age].lat.values
                hist, xedges, yedges = np.histogram2d(
                    test_lons_array, test_lats_array,
                    bins=(lonbins, latbins),
                )
                da = xr.DataArray(
                    hist,
                    dims={
                        "lon":lonbins_da,
                        "lat":latbins_da,
                    },
                    coords={
                        "lon":lonbins_da,
                        "lat":latbins_da,
                    },
                    name=f"{age}-{site}-{start_day}",
                ).expand_dims({
                    "site":[site],
                    "age": [age],
                    "start_day": [start_day],
                    "time": [ds_bins[age].time.values[0]],
                    }
                )
                age_arrays.append(da)
                print(f"s: {site}", f"start_day: {start_day}", f"age: {age}")
            except:
                print("no","s",site, "start_day",start_day, "age:",age)
                break
        try:
            age_data = xr.concat(age_arrays, dim="age")
            daily_arrays.append(age_data)
        except:
            print("no")
            break
    try:
        daily_data = xr.concat(daily_arrays, dim="start_day")
        data_arrays.append(daily_data)
    except:
        print("no")
        break
ds = xr.concat(data_arrays, dim="site")
ds.nbytes/1e9

s: 0 start_day: 150 age: 1
s: 0 start_day: 150 age: 2
s: 0 start_day: 150 age: 3
no s 0 start_day 150 age: 4
s: 0 start_day: 151 age: 1
s: 0 start_day: 151 age: 2
no s 0 start_day 151 age: 3
s: 0 start_day: 152 age: 1
no s 0 start_day 152 age: 2
s: 1 start_day: 150 age: 1
s: 1 start_day: 150 age: 2
s: 1 start_day: 150 age: 3
no s 1 start_day 150 age: 4
s: 1 start_day: 151 age: 1
s: 1 start_day: 151 age: 2
no s 1 start_day 151 age: 3
s: 1 start_day: 152 age: 1
no s 1 start_day 152 age: 2
s: 2 start_day: 150 age: 1
s: 2 start_day: 150 age: 2
s: 2 start_day: 150 age: 3
no s 2 start_day 150 age: 4
s: 2 start_day: 151 age: 1
s: 2 start_day: 151 age: 2
no s 2 start_day 151 age: 3
s: 2 start_day: 152 age: 1
no s 2 start_day 152 age: 2
s: 3 start_day: 150 age: 1
s: 3 start_day: 150 age: 2
s: 3 start_day: 150 age: 3
no s 3 start_day 150 age: 4
s: 3 start_day: 151 age: 1
s: 3 start_day: 151 age: 2
no s 3 start_day 151 age: 3
s: 3 start_day: 152 age: 1
no s 3 start_day 152 age: 2
s: 4 start_day: 

0.0165888

In [10]:
ds.to_netcdf(
    Path(out_path_hist+f"NewConnectivity/ds_pelagicConnectivity_{year}.nc")
)

NameError: name 'ds' is not defined

In [ ]:
# cm = np.zeros((16,16))
# for start_day in range(len(ds.start_day)):
#     for age in range(28):
#         for i in range(16):
#             for j in range(16):
#                 try:
#                     val = ds.isel(site=i, age=age, start_day=start_day).where(
#                         (
#                             (ds.isel(site=i, age=age, start_day=start_day) != 0) &
#                             (ds.isel(site=j, age=age, start_day=start_day) != 0)
#                         )
#                     ).sum()
#                     cm[i,j] += val
#                 except:
#                     print("no")

In [ ]:
# plt.imshow(cm, norm=mcolors.LogNorm())

In [ ]:
# plt.scatter(
#     ds_01[10].lon,
#     ds_01[10].lat,
#     c="r", s=1,
# )
# plt.scatter(
#     ds_06[10].lon,
#     ds_06[10].lat,
#     c="b", s=1,
# )

In [ ]:
# time_mask = ((ds_trajectories.age_sec/(60*60*24)) > 10).compute()
# ds_positions = ds_trajectories.get(["lat", "lon", "age_sec", "time"])
# ds_positions = ds_positions.compute()

In [ ]:
# test_lons = ds_positions.lon
# test_lats = ds_positions.lat
# test_lons_array = test_lons.stack(z=test_lons.dims).dropna(dim="z").values
# test_lats_array = test_lats.stack(z=test_lats.dims).dropna(dim="z").values
# hist, xedges, yedges = np.histogram2d(
#     test_lons_array, test_lats_array,
#     bins=(lonbins, latbins),
# )
# print(hist.nbytes/1e9)
# # np.save(Path(out_path_hist+f"total/hist2d_n{nbins}_{year}_s{start_site_counter:02d}.npy"), hist)
# plt.pcolormesh(hist, norm=mcolors.LogNorm())

In [ ]:
# for month_counter, month in enumerate(range(6,11)):
#     connectivity_matrix = np.zeros((number_sites, number_sites), dtype=int)
#     ds_month_positions = ds_positions.isel(
#         trajectory = slice(month_split[month_counter], month_split[month_counter+1])
#     )
#     connectivity_matrix[start_site_counter,:] = connectivity_function(ds=ds_month_positions, dist=dist)
#     monthly_connectivity_matrices[year, month] = connectivity_matrix

In [ ]:
# np.save(Path(out_path_matrix+f"connectivity_matrices_{year}_s{start_site_counter:02d}.npy"), monthly_connectivity_matrices)

In [ ]:
# ds_variables = ds_trajectories.get(export_vars).expand_dims(expand_dims_dict)
# ds_variables["depth_m"] = (
#     (ds_variables.eta - ds_variables.z * (ds_variables.eta + ds_variables.h0))
# ).compute()
# ds_variables = ds_variables.drop_vars(["eta", "z", "h0"])

In [ ]:
# lonbins = np.linspace(0,30,nbins+1)
# latbins = np.linspace(50,65,nbins+1)

In [ ]:
# Group by months and age
# ds_positions["age_day"] = ds_positions.age_sec/(60*60*24)


In [ ]:
# ds_age = ds_positions.groupby_bins("age_day", np.arange(0,29), labels=np.arange(1,29))

In [ ]:
# ds = ds_positions.drop_vars("time").groupby_bins(
#     "age_sec", 
#     np.arange(0,29*(60*60*24),(60*60*24)),
#     labels=np.arange(1,29)
# )

In [ ]:
# ds = ds_positions.drop_vars("time").groupby_bins(
#     "age_sec", 
#     np.arange(0,29*(60*60*24),(60*60*24)),
#     labels=np.arange(1,29)
# )
# hist_age = {}
# for age in ds:
#     test_lons_array = age[1].lon.values
#     test_lats_array = age[1].lat.values
#     hist, xedges, yedges = np.histogram2d(
#         test_lons_array, test_lats_array,
#         bins=(lonbins, latbins),
#     )
#     hist_age[age[0]] = hist
# np.save(Path(out_path_hist+f"age/hist2d_age_n{nbins}_{year}_s{start_site_counter:02d}.npy"), hist_age)

In [ ]:
# ds_positions = ds_positions.drop_vars("age_sec").where(time_mask, drop = True)

In [ ]:
# ds = ds_positions.where(time_mask, drop = True).groupby("time.month")
# print("group")
# hist_monthly = {}
# for m in ds:
#     test_lons_array = m[1].lon.values
#     test_lats_array = m[1].lat.values
#     hist, xedges, yedges = np.histogram2d(
#         test_lons_array, test_lats_array,
#         bins=(lonbins, latbins),
#     )
#     hist_monthly[m[0]] = hist
#     print(m[0])
# np.save(Path(out_path_hist+f"monthly/hist2d_monthly_n{nbins}_{year}_s{start_site_counter:02d}.npy"), hist_monthly)

In [ ]:
# test_lons = ds_positions.lon
# test_lats = ds_positions.lat

# test_lons_array = test_lons.stack(z=test_lons.dims).dropna(dim="z").values
# test_lats_array = test_lats.stack(z=test_lats.dims).dropna(dim="z").values
# hist, xedges, yedges = np.histogram2d(
#     test_lons_array, test_lats_array,
#     bins=(lonbins, latbins),
# )
# print(hist.nbytes/1e9)
# np.save(Path(out_path_hist+f"total/hist2d_n{nbins}_{year}_s{start_site_counter:02d}.npy"), hist)

In [ ]:
# ds_variables.to_netcdf(
#     path=Path(out_path+f"ds_variables_{year}_s{start_site_counter:02d}.nc"),
#     mode="a",
# )

In [ ]:
# isTest = False
# ds_trajectories = xr.Dataset()
# connectivity_matrix_monthly = {}
# export_vars = ["lat", "lon", "time", "S", "T", "eta", "z", "h0", "age_sec"]
# if isTest == True:
#     month_count = [None, 4,9,13,17,22]
#     years = np.arange(2016,2018)
# else:
#     month_count = [None, 30000,61000,92000,122000,153000]
#     years = np.arange(2016,2025)
# # connectivity_matrix = np.zeros((number_sites, number_sites), dtype=int)
# for year in years:
#     for start_site_counter in site_counter:
#         expand_dims_dict = {"site":[start_site_counter], "year":[year],}
#         file = get_filepath(year, start_site_counter)
#         print("get file", year, start_site_counter)
#         if isTest == True:
#             ds_trajectories = (
#                 xr.open_zarr(file)
#                 .isel(trajectory=slice(None, None, 7000))
#                 .chunk(chunks={'trajectory':-1,'obs':-1,})
#             )
#         else:
#             ds_trajectories = (
#                 xr.open_zarr(file)
#                 .chunk(chunks={'trajectory':100000,'obs':100,})
#             )
#         if start_site_counter == min(site_counter):
#             ds_variables_y = ds_trajectories.get(export_vars).expand_dims(expand_dims_dict)
#             time_mask = ((ds_trajectories.age_sec/(60*60*24)) > 10).compute()
#         else:
#             ds_var_temp = ds_trajectories.get(export_vars).expand_dims(expand_dims_dict)
#             ds_variables_y = xr.combine_by_coords([ds_variables_y,ds_var_temp])
#         ds_trajectories = ds_trajectories.where(time_mask, drop = True)
#         ds_positions = ds_trajectories.get(["lat", "lon"]).compute()
#         for month in np.arange(6,11):
#             temp_connectivity_matrix = np.zeros((number_sites, number_sites), dtype=int)
#             ds_month_positions = ds_positions.isel(trajectory = slice(month_count[month-6], month_count[month-5]))
#             temp_connectivity_matrix[start_site_counter,:]  = connectivity_function(ds=ds_month_positions, dist=dist)
#             if start_site_counter == min(site_counter):
#                 connectivity_matrix_monthly[year, month] = temp_connectivity_matrix
#             else:
#                 connectivity_matrix_monthly[year, month] = connectivity_matrix_monthly[year, month] + temp_connectivity_matrix
#     if year == min(years):
#         ds_variables = ds_variables_y
#     else:
#         ds_variables = xr.combine_by_coords([ds_variables,ds_variables_y])